# Stage 09 / Lecture 61: Custom Tools (`@tool`), `bind_tools()`, and the Manual Tool Execution Loop

This production reference notebook implements and verifies the lower-level tool mechanics beneath modern LangChain agents:
1. **Tool Schema Pairing (`@tool`)**: Decorating Python functions into structured tools exposing `.name`, `.description`, and `.args`.
2. **Tool Binding (`model.bind_tools()`)**: Inspecting the `AIMessage` payload containing `.tool_calls` (showing that binding proposes invocations without local execution).
3. **The Manual Tool Execution Loop**: Executing `tool.invoke(tool_call)` to receive an ID-matched `ToolMessage`, appending it to the conversation history, and synthesizing the final grounded response.


In [1]:
from typing import List, Dict, Any
import os
from dotenv import load_dotenv

from langchain.tools import tool
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage, BaseMessage

load_dotenv()

if not os.getenv("GROQ_API_KEY"):
    raise EnvironmentError("GROQ_API_KEY must be configured in .env for tool-binding benchmarks.")

model = ChatGroq(model="openai/gpt-oss-20b", temperature=0.0)
print("Model initialized successfully.")


Model initialized successfully.


### 1. Creating a Custom Tool with `@tool` and Inspecting Its Schema

A LangChain tool pairs an execution callable with a JSON-compatible metadata schema derived directly from the function name, type annotations, and docstring.


In [2]:
@tool
def get_weather(location: str) -> str:
    """Get the weather at a location."""
    return f"It is sunny in {location}."


assert get_weather.name == "get_weather"
assert "weather at a location" in get_weather.description.lower()
assert "location" in get_weather.args

print(f"Tool Name: {get_weather.name}")
print(f"Tool Description: {get_weather.description}")
print(f"Tool Args Schema: {get_weather.args}")


Tool Name: get_weather
Tool Description: Get the weather at a location.
Tool Args Schema: {'location': {'title': 'Location', 'type': 'string'}}


### 2. Binding Tools (`model.bind_tools`) & Executing the Manual Loop

Calling `model_with_tools.invoke(messages)` only produces an `AIMessage` requesting a tool invocation (`.tool_calls`). The application orchestrator must execute `get_weather.invoke(tool_call)`—which automatically wraps the output in a `ToolMessage` preserving `tool_call_id`—and re-invoke the model.


In [3]:
model_with_tools = model.bind_tools([get_weather])

# Step 1: Initialize conversation with system instructions and user query
messages: List[BaseMessage] = [
    SystemMessage(content="You are a concise assistant. Answer using plain text without emojis."),
    HumanMessage(content="What is the weather like in Boston?")
]

# Step 2: Model proposes tool call(s)
ai_msg: AIMessage = model_with_tools.invoke(messages)
messages.append(ai_msg)

assert isinstance(ai_msg, AIMessage)
assert len(ai_msg.tool_calls) == 1
assert ai_msg.tool_calls[0]["name"] == "get_weather"
assert ai_msg.tool_calls[0]["args"]["location"].lower() == "boston"

print("Step 1 - Proposed Tool Call:")
print(f"  Tool: {ai_msg.tool_calls[0]['name']} | Args: {ai_msg.tool_calls[0]['args']} | ID: {ai_msg.tool_calls[0]['id']}")

# Step 3: Execute each requested tool call using tool.invoke(tool_call)
for tc in ai_msg.tool_calls:
    tool_result: ToolMessage = get_weather.invoke(tc)
    assert isinstance(tool_result, ToolMessage)
    assert tool_result.tool_call_id == tc["id"]
    messages.append(tool_result)
    print(f"Step 2 - Executed ToolMessage: '{tool_result.content}' (linked to ID: {tool_result.tool_call_id})")

# Step 4: Synthesize final response from full context
final_response: AIMessage = model_with_tools.invoke(messages)
messages.append(final_response)

assert isinstance(final_response, AIMessage)
assert len(final_response.tool_calls) == 0
assert "sunny" in final_response.content.lower() and "boston" in final_response.content.lower()

print(f"Step 3 - Final Grounded Answer: {final_response.content}")
print(f"\nTotal messages in manual execution trajectory: {len(messages)}")


Step 1 - Proposed Tool Call:
  Tool: get_weather | Args: {'location': 'Boston'} | ID: fc_2e603716-6f16-47e6-808e-e0d7fa4abc90
Step 2 - Executed ToolMessage: 'It is sunny in Boston.' (linked to ID: fc_2e603716-6f16-47e6-808e-e0d7fa4abc90)


Step 3 - Final Grounded Answer: It is sunny in Boston.

Total messages in manual execution trajectory: 5


### 3. Production Pattern: Multi-Tool Registry Dispatch

When binding multiple tools, an orchestrator maps tool names via a dictionary registry (`{t.name: t for t in tools}`) so any dynamic `tool_call` is safely routed and executed.


In [4]:
@tool
def get_exchange_rate(currency_pair: str) -> str:
    """Get the live exchange rate for a currency pair (e.g. USD/EUR)."""
    rates = {"USD/EUR": "0.92", "USD/GBP": "0.79"}
    rate = rates.get(currency_pair.upper(), "1.00")
    return f"The exchange rate for {currency_pair.upper()} is {rate}."


tools_list = [get_weather, get_exchange_rate]
tool_registry = {t.name: t for t in tools_list}

multi_tool_model = model.bind_tools(tools_list)

query_messages: List[BaseMessage] = [
    SystemMessage(content="You are a concise assistant. Plain text only without emojis."),
    HumanMessage(content="What is the exchange rate for USD/EUR?")
]

proposed_msg = multi_tool_model.invoke(query_messages)
query_messages.append(proposed_msg)

assert len(proposed_msg.tool_calls) == 1
chosen_name = proposed_msg.tool_calls[0]["name"]
assert chosen_name == "get_exchange_rate"

executed_tool_msg = tool_registry[chosen_name].invoke(proposed_msg.tool_calls[0])
query_messages.append(executed_tool_msg)

grounded_final = multi_tool_model.invoke(query_messages)
assert "0.92" in grounded_final.content

print(f"Dispatched Tool via Registry: {chosen_name}")
print(f"Final Synthesized Output: {grounded_final.content}")
print("\nAll Stage 09 / Lecture 61 tool creation and manual loop assertions passed.")


Dispatched Tool via Registry: get_exchange_rate
Final Synthesized Output: The exchange rate for USD/EUR is 0.92.

All Stage 09 / Lecture 61 tool creation and manual loop assertions passed.
